In [5]:
import pandas as pd
import numpy as np

import plotly.express as px
import importlib

from Irina import utility_functions as uf

In [6]:
domain_id_list = uf.df_topics.domain_id.drop_duplicates().sort_values().to_list()
domain_list = []
for domain in domain_id_list:
    field_id_list = uf.df_topics.query(f"domain_id == {domain}").field_id.drop_duplicates().sort_values().to_list()
    field_list = []
    for field in field_id_list:
        subfield_id_list = uf.df_topics.query(f"field_id == {field}").subfield_id.drop_duplicates().sort_values().to_list()
        subfield_list = [{"id": subfield, "length": 0.01} for subfield in subfield_id_list]
        field_list.append({
            "id": field,
            "length": 0.1,
            "branches": subfield_list,
        })
    domain_list.append({
        "id": domain,
        "length": 1,
        "branches": field_list,
    })

subfields_tree = {
    "id": 0,
    "branches": domain_list,
}

# subfields_tree

# Read metrics_data

In [7]:
path = "../data/"
df_country_subfield_norm_world_norm = pd.read_csv(path+"df_country_subfield_norm_world_norm.csv", index_col="country")
df_country_subfield_norm = pd.read_csv(path+"df_country_subfield_norm.csv", index_col="country")

# W1 - distance

In [8]:
def get_dist_w1_tree(tree, mu_dict, nu_dict):
    subtree = tree.get("branches", None)
    if subtree is None:
        leave_id = tree["id"]
        edge_length = tree.get("length", None)
        mu_id = mu_dict.get(str(leave_id), 0)
        nu_id = nu_dict.get(str(leave_id), 0)
        return mu_id, nu_id, abs(mu_id - nu_id) * edge_length
    else:
        mu_id_array = np.full(len(subtree), 0, dtype=float)
        nu_id_array = np.full(len(subtree), 0, dtype=float)
        dist_sum = 0
        edge_length = tree.get("length", None)
        for i, branch in enumerate(subtree):
            mu_id_array[i], nu_id_array[i], dist_branch = get_dist_w1_tree(branch, mu_dict, nu_dict)
            dist_sum += dist_branch
        if edge_length is None:
            return mu_id_array.sum(), nu_id_array.sum(), dist_sum
        else:
            return mu_id_array.sum(), nu_id_array.sum(), dist_sum + abs(mu_id_array.sum() - nu_id_array.sum()) * edge_length


In [9]:
get_dist_w1_tree(subfields_tree,
                 mu_dict = df_country_subfield_norm_world_norm.loc["RU"].dropna().to_dict(),
                 nu_dict = df_country_subfield_norm_world_norm.loc["UA"].dropna().to_dict())

(np.float64(0.9999999999999887),
 np.float64(0.9999999999999893),
 np.float64(0.18419546607979043))

# Get country distances

In [10]:
country_list = df_country_subfield_norm_world_norm.index.to_list()
top_20_countries_list = uf.top_n_countries_by_articles(20)
top_20_idx = [country_list.index(c) for c in top_20_countries_list]

In [11]:
country_dist_w1 = np.full((len(country_list), len(country_list)), np.nan, dtype=float)

for i, country1 in enumerate(country_list):
    for j, country2 in enumerate(country_list):
        if i == j:
            country_dist_w1[i, j] = 0
        if i < j:
            _, _, country_dist_w1[i, j] = _, _, country_dist_w1[j, i] = get_dist_w1_tree(
                subfields_tree,
                mu_dict = df_country_subfield_norm.loc[country1].dropna().to_dict(),
                nu_dict = df_country_subfield_norm.loc[country2].dropna().to_dict())
df_country_dist_w1 = pd.DataFrame(country_dist_w1, index=country_list, columns=country_list)

In [12]:
country_dist_w1_world = np.full((len(country_list), len(country_list)), np.nan, dtype=float)

for i, country1 in enumerate(country_list):
    for j, country2 in enumerate(country_list):
        if i == j:
            country_dist_w1_world[i, j] = 0
        if i < j:
            _, _, country_dist_w1_world[i, j] = _, _, country_dist_w1_world[j, i] = get_dist_w1_tree(
                subfields_tree,
                mu_dict = df_country_subfield_norm_world_norm.loc[country1].dropna().to_dict(),
                nu_dict = df_country_subfield_norm_world_norm.loc[country2].dropna().to_dict())
df_country_dist_w1_world = pd.DataFrame(country_dist_w1_world, index=country_list, columns=country_list)

In [13]:
# df_country_dist_w1.to_csv(path+"df_country_dist_w1.csv")
# df_country_dist_w1_world.to_csv(path+"df_country_dist_w1_world_norm.csv")

In [14]:
df_country_dist_w1

,AD,AE,AF,AG,AL,AM,AO,AR,AS,AT,...,VG,VI,VN,VU,WS,XK,YE,ZA,ZM,ZW
AD,0.000000,0.793953,0.863889,1.845000,0.842173,1.239060,0.660635,0.560704,0.886667,0.875824,...,1.434297,0.711667,0.993301,0.868333,0.671333,0.928000,0.891347,0.600792,1.221405,0.779365
AE,0.793953,0.000000,0.621796,1.465251,0.343745,0.806816,0.473339,0.305023,0.477111,0.336940,...,1.093107,0.479932,0.367087,0.575976,0.794974,0.276961,0.359352,0.342143,0.901962,0.449335
AF,0.863889,0.621796,0.000000,1.107222,0.354623,1.232280,0.262778,0.449091,0.628889,0.470927,...,1.527387,0.702500,0.687737,0.276111,0.681889,0.612794,0.669595,0.388670,0.452694,0.353984
AG,1.845000,1.465251,1.107222,0.000000,1.250625,1.826754,1.300754,1.491588,1.420000,1.335060,...,2.075325,1.640833,1.421749,1.110417,1.625000,1.422714,1.517450,1.378336,0.725959,1.242798
AL,0.842173,0.343745,0.354623,1.250625,0.000000,0.933708,0.303363,0.339315,0.444375,0.217197,...,1.243913,0.507381,0.370957,0.490565,0.838750,0.308869,0.406298,0.319685,0.583731,0.342117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK,0.928000,0.276961,0.612794,1.422714,0.308869,0.701484,0.494286,0.433719,0.618381,0.255846,...,0.998077,0.352738,0.199283,0.686667,0.917714,0.000000,0.216350,0.437524,0.789874,0.548063
YE,0.891347,0.359352,0.669595,1.517450,0.406298,0.617304,0.566318,0.397866,0.784437,0.242392,...,0.920725,0.320171,0.142141,0.850684,0.895563,0.216350,0.000000,0.501985,0.879834,0.697374
ZA,0.600792,0.342143,0.388670,1.378336,0.319685,1.070279,0.225622,0.212481,0.416669,0.382040,...,1.377771,0.593006,0.505150,0.379846,0.559518,0.437524,0.501985,0.000000,0.694924,0.211556
ZM,1.221405,0.901962,0.452694,0.725959,0.583731,1.421618,0.629431,0.817845,0.905459,0.709766,...,1.712625,0.978203,0.863107,0.543892,0.956216,0.789874,0.879834,0.694924,0.000000,0.625263


In [16]:

importlib.reload(uf)

<module 'utility_functions' from '/Users/irinavorobeva/PycharmProjects/geoscience/utility_functions.py'>

In [17]:
uf.plotly_heatmap(df_country_dist_w1_world,
                  # x_labels=df_country_dist_w1_world.index.astype(str).to_list(),
                  # y_labels=df_country_dist_w1_world.columns.astype(str).to_list(),
                  x_labels=uf.top_n_countries_by_articles(20),
                  y_labels=uf.top_n_countries_by_articles(20),
                  x_type="country",
                  y_type="country",
                  x_name="Country (x)",
                  y_name="Country (y)",
                  z_name="Distance",
                  title="W1 distance",
                  line_height=40)

In [20]:
uf.get_country_info("SS")

,name,region,sub-region,code
208,South Sudan,Africa,Sub-Saharan Africa,SS


In [33]:
import numpy as np
import plotly.graph_objects as go

country_list = ["CA", "ES", "GB", "BR"]  # list of countries
x = df_country_subfield_norm.columns.to_list()

fig = go.Figure()

for country in country_list:
    y = df_country_subfield_norm.loc[country].values
    y_safe = [v if not np.isnan(v) else None for v in y]  # handle NaNs

    fig.add_trace(go.Scatter(
        x=x,
        y=y_safe,
        mode="lines",
        name=f"{uf.id2name_country[country]}"
    ))

fig.update_layout(
    title="Interest distributions for selected countries",
    xaxis=dict(title="Sub-field"),
    yaxis=dict(title="Sub-field share"),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=60, r=60, t=40, b=40)
)

fig.show()

In [34]:
# country = "AX"
# top_n = 20
# df_plot = df_country_dist_w1.sort_values(country, ascending=True)[[country]].head(top_n)
# px.bar(df_plot, x=country, title=uf.get_country_info(country).iloc[0, 0])

In [35]:
# top_n = 10
# countries = uf.top_n_countries_by_articles(top_n)
#
# # Create figure
# fig = go.Figure()
#
# for country in countries:
#     df_plot = df_country_dist_w1.sort_values(country, ascending=True).head(top_n)
#     fig.add_trace(
#         go.Bar(
#             x=df_plot[country],           # value
#             y=np.arange(top_n),
#             hovertext=[code+" "+uf.id2name_country[code] for code in df_plot.index],# category
#             name=uf.get_country_info(country).iloc[0, 0],
#             orientation='h'
#         )
#     )
#
# # Layout
# fig.update_layout(
#     barmode='group',       # 'stack' if you prefer stacked bars
#     title=f"Top {top_n} categories for selected countries",
#     xaxis_title="Value",
#     yaxis_title=None,
#     height=800,
#     margin=dict(l=150),    # leave space for long y labels
# )
#
# # Optional: largest bar at top
# fig.update_yaxes(autorange="reversed")
#
# fig.show()

In [36]:
df_country_dist_w1_stats = (
    df_country_dist_w1
    .mean()
    .to_frame("mean")
    .assign(median=df_country_dist_w1.median(),
            std=df_country_dist_w1.std(),
            range=df_country_dist_w1.max() - df_country_dist_w1.min(),
            gini=df_country_dist_w1.apply(uf.gini, axis=1),)
)
df_country_dist_w1_stats

,mean,median,std,range,gini
AD,0.943706,0.881513,0.304014,2.220000,0.166231
AE,0.628556,0.519835,0.354424,1.855040,0.303224
AF,0.668498,0.625287,0.309816,1.720000,0.252225
AG,1.364596,1.417885,0.353323,2.220000,0.137910
AL,0.574268,0.503392,0.327084,1.768125,0.305536
...,...,...,...,...,...
XK,0.605542,0.519044,0.360344,1.818095,0.319603
YE,0.622934,0.570609,0.388202,1.906093,0.336376
ZA,0.602698,0.557624,0.334905,1.782639,0.299226
ZM,0.840431,0.823823,0.315311,1.869297,0.206709


In [39]:
px.histogram(df_country_dist_w1_stats, x="std")